In [1]:
import sys
sys.path.insert(0, 'src')

# -- Configuration: edit these for your machine
RESOURCES_DIR = 'resources'
CUELIST_PATH  = 'example_cuelist.csv'

In [2]:
from midi import MidiOut
print('Available MIDI ports:')
for p in MidiOut.list_ports():
    print(' ', p)

Available MIDI ports:
  IAC Driver Bus 1
  IAC Driver Bus 2
  Scarlett 2i4 USB
  YAMAHA MOTIF6 Port1
  YAMAHA MOTIF6 Port2
  YAMAHA MOTIF6 Port3
  YAMAHA MOTIF6 Port4
  YAMAHA MOTIF6 Port5
  YAMAHA MOTIF6 Port6
  YAMAHA MOTIF6 Port7
  YAMAHA MOTIF6 Port8
  to Max 1
  to Max 2


In [3]:
from catalog import Catalog
from cuelist import load_cuelist
from midi import MidiOut
from state import PerformanceState
from ui import build_ui
from IPython.display import display

catalog  = Catalog(RESOURCES_DIR)
cues     = load_cuelist(CUELIST_PATH, catalog)
midi_out = MidiOut()
state    = PerformanceState(cues=cues, catalog=catalog, midi_out=midi_out)
widget   = build_ui(state, midi_out)
display(widget)

In [5]:
# -- Cuelist generator (edit and re-run at any time) ------------------
#
# filter_scales  – match scale names containing any of the given strings
# filter_voices  – narrow by bank and/or category_abbr / category_1
# generate_cuelist – pair them via "random", "cross_product", or "zip"
# write_cuelist + state.reload() – hot-swap without restarting the widget
#
# category_abbr quick reference:
#   Ap acoustic piano   Pd pad        St strings    Br brass
#   Or organ            Gt guitar     Ba bass        Ld lead
#   Cp chromatic perc   Se sfx        Kb keyboard    Dr drums

from cuelist_gen import filter_scales, filter_voices, generate_cuelist, write_cuelist
from cuelist import load_cuelist

scales = filter_scales(catalog, contains=['DEGUNG', 'PELOG', 'HIRAJOSHI', 'SLENDRO'])
# voices = filter_voices(catalog, banks=['PRE2', 'PRE3'], category_abbr=['Pd', 'St'])
voices = filter_voices(catalog, banks=['PRE1'], category_abbr=['Kb'])

pairs  = generate_cuelist(scales, voices, n=16, seed=None)
write_cuelist(CUELIST_PATH, pairs)
print(f"Wrote {len(pairs)} cues  ({len(scales)} scales × {len(voices)} voices available)")
for s, v in pairs:
    print(f"  {s.name:<22} {v.bank}:{v.name}")

# Hot-reload into the running widget (no-op if widget not yet started):
try:
    state.reload(load_cuelist(CUELIST_PATH, catalog))
    print("State reloaded.")
except NameError:
    print("(Run main cell first to enable live reload.)")

Wrote 16 cues  (56 scales × 32 voices available)
  SLENDROC1              PRE1:80th_Boost
  SLENDROC1              PRE1:Rich_FM
  SLENDRO4               PRE1:Yama_EP's
  SLENDRO2               PRE1:Max_Tine
  SLENDRO4               PRE1:Ice_Piano
  SLENDRO_UDAN           PRE1:TX802
  PELOG18                PRE1:Yama_EP's
  SLENDRO5_4             PRE1:WurliAmped
  SLENDROC1              PRE1:TouchClavi
  SLENDROB3              PRE1:Yama_EP's
  PELOG18                PRE1:Early_70's
  DEGUNG1                PRE1:DynoStrait
  SLENDRO5_4             PRE1:SuperClavi
  SLENDRO5_2             PRE1:WurliTrem
  SLENDRO_Q13            PRE1:Early_70's
  SLENDRO6               PRE1:PulseClavi
State reloaded.


In [5]:
import mido
print(mido.get_output_names())
with mido.open_output('YAMAHA MOTIF6 Port1') as port:
    print(f"Port opened: {port}")
    msg = mido.Message('program_change', program=0)
    port.send(msg)
    print(f"Sent: {msg}")

['IAC Driver Bus 1', 'IAC Driver Bus 2', 'Scarlett 2i4 USB', 'YAMAHA MOTIF6 Port1', 'YAMAHA MOTIF6 Port2', 'YAMAHA MOTIF6 Port3', 'YAMAHA MOTIF6 Port4', 'YAMAHA MOTIF6 Port5', 'YAMAHA MOTIF6 Port6', 'YAMAHA MOTIF6 Port7', 'YAMAHA MOTIF6 Port8', 'to Max 1', 'to Max 2']
Port opened: <open output 'YAMAHA MOTIF6 Port1' (RtMidi/MACOSX_CORE)>
Sent: program_change channel=0 program=0 time=0


In [7]:
msgs = catalog.sysex_for_voice(state.current_voice().bank, state.current_voice().index)
print(f"Sending {len(msgs)} sysex messages via {midi_out.port_name!r}")
midi_out.send_sysex(msgs)

Sending 72 sysex messages via 'YAMAHA MOTIF6 Port1'
